In [1]:
import cv2
import matplotlib.pyplot as plt
from sys import platform
from pathlib import Path
import os
import imageio.v3 as iio
from time import sleep
import numpy as np

In [2]:
def plot_mri(img, save_img=False, save_path=None, grays=True):
    if grays:
        plt.imshow(img, cmap='Grays_r')
    else:
        plt.imshow(img)
    plt.axis('off')
    plt.margins(x=0)
    plt.subplots_adjust(top=1, bottom=0, right=1, left=0, hspace=0, wspace=0)
    if save_img:
        assert(save_path is not None), 'No path provided'
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    else:
        plt.show()
    plt.close()

#### System Paths

In [3]:
if platform == 'linux':
    home_dir = '~'
elif platform == 'win32':
    home_dir = '%HOMEDRIVE%%HOMEPATH%'
else:
    raise ValueError("Cannot set home directory for this OS.")
print(home_dir)

%HOMEDRIVE%%HOMEPATH%


#### Output directories

In [4]:
img_dir = r'OneDrive\Pictures\visible-human' #Stupid OneDrive

In [5]:
# Working directory
proj_dir = r'C:\Users\th1si\OneDrive\Pictures\visible-human' # From ./human-visibility-dl.ipynb
raw_dir = rf'{proj_dir}\raw' # From ./human-visibility-dl.ipynb

# Clean directory
clean_dir = rf'{proj_dir}\clean'
mk_clean_dir = Path(clean_dir)
mk_clean_dir.mkdir(parents=True, exist_ok=True)

print(f'Raw dir: {raw_dir}')
print(f'Clean dir: {clean_dir}')

Raw dir: C:\Users\th1si\OneDrive\Pictures\visible-human\raw
Clean dir: C:\Users\th1si\OneDrive\Pictures\visible-human\clean


#### Get image paths

##### Select body part (all, abdomen, thorax, head, pelvis, thighs, legs) 

In [6]:
body_part = 'thorax'

In [7]:
extension = '.png'
raw_imgs = []
for root, dirs, files in os.walk(raw_dir):
    if body_part != 'all':
        dirs[:] = [d for d in dirs if d == body_part]
    for file in files:
        if file.endswith(extension):
            raw_imgs.append(os.path.join(root, file))
print(f'{len(raw_imgs)} images ready. Example: ')
print(raw_imgs[:5])

409 images ready. Example: 
['C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\raw\\thorax\\a_vm1280.png', 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\raw\\thorax\\a_vm1281.png', 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\raw\\thorax\\a_vm1282.png', 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\raw\\thorax\\a_vm1283.png', 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\raw\\thorax\\a_vm1284.png']


#### Clean images (crop, mask)
Original shape (1216, 2048, 3)

##### Force 16:9 ratio

In [8]:
# New size: 16:9 ratio by const
y_start=44
y_end=1015
x_start=149
x_end=1876

w = x_end-x_start
h = y_end-y_start
print(f'Width: {w}, Height: {h}, Ratio: {w/h}')

Width: 1727, Height: 971, Ratio: 1.7785787847579815


In [9]:
# Body part directory
body_part_dir = rf'{clean_dir}\{body_part}'
temp_mk_dir = Path(body_part_dir)
temp_mk_dir.mkdir(parents=True, exist_ok=True)
print(f'Saving images in {body_part_dir}')

Saving images in C:\Users\th1si\OneDrive\Pictures\visible-human\clean\thorax


Actual cleaning

In [10]:
c = 0
for img_path in raw_imgs:
    save_path = rf'{body_part_dir}\{img_path.split('\\')[-1]}'
    img = cv2.imread(img_path)
    clean_img = img[y_start:y_end, x_start:x_end]

    # img = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB)
    
    # Convert BGR to HSV (beter for masking)
    hsv = cv2.cvtColor(clean_img, cv2.COLOR_BGR2HSV)
    lower_blue = np.array([50, 0, 0]) # hue, saturation, brigthness values from 
    upper_blue = np.array([179, 255, 255])
    mask = cv2.inRange(hsv, lower_blue, upper_blue)
  
    # Blur mask
    kernel_size = 1
    feathered_mask = cv2.GaussianBlur(mask, (kernel_size, kernel_size), 0)

    clean_img[feathered_mask > 0] = [0, 0, 0]

    
    cv2.imwrite(save_path, clean_img)
    med_img = np.median(range(len(raw_imgs)))
    if c == med_img:
        plot_mri(clean_img)

#### Animate segmentation
Takes a while and generates a 2.6Gb gif, see other script for other methods.

In [11]:
sorted_imgs = {}
for root, dirs, files in os.walk(body_part_dir):
    for file in files:
        k = int(''.join([c for c in file if c.isdigit()]))
        v = os.path.join(root, file)
        sorted_imgs[k] = v
sorted_imgs = {k: sorted_imgs[k] for k in sorted(sorted_imgs)} # Sort images 
[sorted_imgs[k] for k in list(sorted_imgs.keys())[:5]] # Verify proper order for axial segmentation

['C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\clean\\thorax\\a_vm1280.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\clean\\thorax\\a_vm1281.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\clean\\thorax\\a_vm1282.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\clean\\thorax\\a_vm1283.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\clean\\thorax\\a_vm1284.png']

In [12]:
filenames = [sorted_imgs[k] for k in sorted_imgs] 
images = [iio.imread(f) for f in filenames]
iio.imwrite(f'{clean_dir}\\gif-{body_part}-axial.gif', images, duration=25, loop=0)
print('Done.')

Done.
